In [8]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


df = pd.read_csv("../data/student_tickets.csv")

X = df["ticket_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("classifier", MultinomialNB())
    ]),

    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("classifier", LinearSVC())
    ]),

    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(y_test, predictions))

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

print("\nModel Comparison:")
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

joblib.dump(best_model, "../outputs/best_ticket_model.joblib")


print(f"\nBest model saved: {best_model_name}")


Logistic Regression
                     precision    recall  f1-score   support

  academic_advising       0.60      0.60      0.60         5
         admissions       0.80      0.80      0.80         5
course_registration       0.57      0.80      0.67         5
      financial_aid       0.67      0.40      0.50         5
         graduation       1.00      0.60      0.75         5
  technical_support       0.62      0.83      0.71         6

           accuracy                           0.68        31
          macro avg       0.71      0.67      0.67        31
       weighted avg       0.71      0.68      0.67        31


Naive Bayes
                     precision    recall  f1-score   support

  academic_advising       0.60      0.60      0.60         5
         admissions       0.75      0.60      0.67         5
course_registration       0.57      0.80      0.67         5
      financial_aid       0.67      0.40      0.50         5
         graduation       0.75      0.60      0